# RootGuard - AI-Powered Rootkit Detection

This notebook follows the actual project idea: detecting rootkit risk using binary security flags and numeric system signals.

The problem is **binary classification**:

- `normal`
- `rootkit`

The notebook trains separate models and then builds a stacked ensemble.

## Latest Run Output

The model comparison script was run successfully on the current dataset.

| Model | Accuracy | Precision | Recall | F1-score |
|---|---:|---:|---:|---:|
| Decision Tree | 100.00% | 100.00% | 100.00% | 100.00% |
| Random Forest | 100.00% | 100.00% | 100.00% | 100.00% |
| Gradient Boosting | 100.00% | 100.00% | 100.00% | 100.00% |
| SVM | 100.00% | 100.00% | 100.00% | 100.00% |
| Logistic Regression | 100.00% | 100.00% | 100.00% | 100.00% |
| Stacked Ensemble | 100.00% | 100.00% | 100.00% | 100.00% |

Best model from this run: **Decision Tree**. The saved model artifact is `models/rootguard_pipeline.joblib`.

Note: This is a controlled mock/synthetic feature dataset for a college project demonstration, so very high accuracy is expected.

## 1. Import Libraries

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from src.rootguard.config import CATEGORICAL_FEATURES, MODEL_PATH, NUMERIC_FEATURES, TARGET_COLUMN
from src.rootguard.data import generate_synthetic_incidents, load_sample_data
from src.rootguard.features import validate_features

ModuleNotFoundError: No module named 'joblib'

## 2. Load Rootkit Feature Dataset

In [ ]:
realistic_mock_data = load_sample_data()
synthetic_data = generate_synthetic_incidents(n_samples=1200, random_state=42)
data = pd.concat([synthetic_data, realistic_mock_data], ignore_index=True)

print("Dataset shape:", data.shape)
data.head()

In [ ]:
data[TARGET_COLUMN].value_counts()

## 3. Train-Test Split

In [ ]:
X = validate_features(data)
y = data[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

results = []
trained_models = {}

## 4. Helper Function for Evaluation

In [ ]:
def evaluate_model(model_name, classifier):
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, pos_label="rootkit")
    recall = recall_score(y_test, predictions, pos_label="rootkit")
    f1 = f1_score(y_test, predictions, pos_label="rootkit")

    trained_models[model_name] = pipeline
    results.append({
        "model": model_name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
    })
    print(f"{model_name} Accuracy: {accuracy:.2%}")
    return pipeline

## 5. Decision Tree Classifier

In [ ]:
decision_tree_model = evaluate_model(
    "Decision Tree",
    DecisionTreeClassifier(max_depth=10, class_weight="balanced", random_state=42),
)

## 6. Random Forest Classifier

In [ ]:
random_forest_model = evaluate_model(
    "Random Forest",
    RandomForestClassifier(
        n_estimators=250,
        max_depth=12,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
    ),
)

## 7. Gradient Boosting Classifier

In [ ]:
gradient_boosting_model = evaluate_model(
    "Gradient Boosting",
    GradientBoostingClassifier(random_state=42),
)

## 8. Support Vector Machine Classifier

In [ ]:
svm_model = evaluate_model(
    "SVM",
    SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42),
)

## 9. Logistic Regression Classifier

In [ ]:
logistic_regression_model = evaluate_model(
    "Logistic Regression",
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)

## 10. Stacked Ensemble Classifier

The stacked model combines Decision Tree, Random Forest, Gradient Boosting, and SVM. Logistic Regression is the final estimator.

In [ ]:
stacked_model = evaluate_model(
    "Stacked Ensemble",
    StackingClassifier(
        estimators=[
            ("dt", DecisionTreeClassifier(max_depth=10, random_state=42)),
            ("rf", RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)),
            ("gb", GradientBoostingClassifier(random_state=42)),
            ("svm", SVC(kernel="rbf", probability=True, random_state=42)),
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=5,
    ),
)

## 11. Accuracy Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
results_df

## 12. Best Model Report

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

print("Best model:", best_model_name)
print(classification_report(y_test, best_predictions))
pd.DataFrame(confusion_matrix(y_test, best_predictions), index=["actual_normal", "actual_rootkit"], columns=["pred_normal", "pred_rootkit"])

## 13. Save Best Model

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODEL_PATH)
print("Saved model to:", MODEL_PATH)